# 🧩 Mini-Lab: Response Caching

**Module 10: Production Deployment** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why caching LLM responses dramatically reduces cost and latency
2. **Implement** an in-memory cache using a dictionary with hash-based keys
3. **Compare** cached vs. uncached response times to see the performance benefit
4. **Recognize** key caching design decisions: what to use as the cache key and when to invalidate

## Target Concepts

| Concept | Description |
|---------|-------------|
| Caching Strategies | Storing LLM responses so identical requests are served instantly from cache instead of making a new (slow, expensive) API call |

## Setup

In [2]:
import hashlib
import json
import time
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

client = OpenAI()
MODEL = "gpt-4o-mini"
print("✅ Ready")

✅ Ready


## Why Cache LLM Responses?

Every LLM API call costs **money** and takes **time** (typically 0.5–3 seconds). Many applications send the same or very similar requests repeatedly:

- A FAQ chatbot answering the same questions
- A classification endpoint receiving duplicate inputs
- Retried requests from frontend clients

Caching stores the response for a given request so that **the second time the same request arrives, we return the stored result instantly** — no API call, no cost, no latency.

```
First request:   Client → Cache MISS → LLM API → Store in cache → Return  (~1.5s)
Repeat request:  Client → Cache HIT  → Return from cache               (~0.001s)
```

## Step 1 — Build a Simple LLM Cache

The core idea:
1. **Create a cache key** from the request parameters (message + temperature + model)
2. **Check** if the key exists in the cache before calling the LLM
3. **Store** new results in the cache after each API call

We use a SHA-256 hash of the request parameters as the key. This ensures consistent, fixed-length keys regardless of prompt length.

In [18]:
class LLMCache:
    """Simple in-memory cache for LLM responses."""

    def __init__(self):
        self._cache = {}  # key → response string
        self.hits = 0
        self.misses = 0

    def _make_key(self, message: str, temperature: float, model: str) -> str:
        """Create a deterministic cache key from request parameters."""
        raw = json.dumps(
            {"message": message, "temperature": temperature, "model": model},
            sort_keys=True,
        )
        return hashlib.sha256(raw.encode()).hexdigest()

    def get(self, message: str, temperature: float, model: str) -> str | None:
        """Look up a cached response. Returns None on cache miss."""
        key = self._make_key(message, temperature, model)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        self.misses += 1
        return None

    def set(self, message: str, temperature: float, model: str, response: str):
        """Store a response in the cache."""
        key = self._make_key(message, temperature, model)
        self._cache[key] = response

    @property
    def size(self) -> int:
        return len(self._cache)

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0


cache = LLMCache()
print(f"✅ Cache initialized (size: {cache.size})")

✅ Cache initialized (size: 0)


## Step 2 — Cached LLM Call Function

Now we wrap the OpenAI call with cache logic: **check cache first → call LLM only on miss → store result**.

In [19]:
def chat_with_cache(message: str, temperature: float = 0.0) -> dict:
    """
    Call the LLM with caching. Returns a dict with:
      - response: the LLM's answer
      - cached: whether this was a cache hit
      - elapsed: time taken in seconds
    """
    start = time.perf_counter()

    # 1. Check cache
    cached_response = cache.get(message, temperature, MODEL)
    if cached_response is not None:
        elapsed = time.perf_counter() - start
        return {"response": cached_response, "cached": True, "elapsed": elapsed}

    # 2. Cache miss — call the LLM
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": message}],
        temperature=temperature,
    )
    response_text = completion.choices[0].message.content

    # 3. Store in cache
    cache.set(message, temperature, MODEL, response_text)

    elapsed = time.perf_counter() - start
    return {"response": response_text, "cached": False, "elapsed": elapsed}


print("✅ chat_with_cache() defined")

✅ chat_with_cache() defined


## Step 3 — See Caching in Action

Let's send the **same prompt twice** and compare what happens.

In [20]:
prompt = "What are the three main benefits of caching? Reply in a short bullet list."

# First call — cache MISS (calls the LLM)
result1 = chat_with_cache(prompt)
print(f"Call 1 — Cached: {result1['cached']}  |  Time: {result1['elapsed']:.4f}s")

# Second call — cache HIT (instant)
result2 = chat_with_cache(prompt)
print(f"Call 2 — Cached: {result2['cached']}  |  Time: {result2['elapsed']:.4f}s")

# Speedup
speedup = result1['elapsed'] / result2['elapsed'] if result2['elapsed'] > 0 else float('inf')
print(f"\n⚡ Cache hit was {speedup:,.0f}x faster")
print()
md(result1['response'])

Call 1 — Cached: False  |  Time: 2.6711s
Call 2 — Cached: True  |  Time: 0.0001s

⚡ Cache hit was 41,477x faster



- **Improved Performance**: Caching reduces latency and speeds up data retrieval by storing frequently accessed data closer to the application or user.
- **Reduced Load on Backend Systems**: By serving cached data, it decreases the number of requests to the database or server, minimizing resource consumption and improving overall system efficiency.
- **Cost Efficiency**: Caching can lower operational costs by reducing the need for expensive hardware and bandwidth, as it optimizes resource usage and decreases response times.

## Step 4 — Cache Key Sensitivity

An important design decision: **what makes two requests "the same"?**

Our cache key includes `message`, `temperature`, and `model`. This means:
- Same message + same temperature → cache **HIT** ✅
- Same message + different temperature → cache **MISS** ❌

Let's verify this.

In [21]:
prompt = "What is an API? Reply in one sentence."

# Call with temperature=0.0
r1 = chat_with_cache(prompt, temperature=0.0)
print(f"temp=0.0  →  Cached: {r1['cached']}  |  Time: {r1['elapsed']:.4f}s")

# Same prompt, temperature=0.0 again → HIT
r2 = chat_with_cache(prompt, temperature=0.0)
print(f"temp=0.0  →  Cached: {r2['cached']}  |  Time: {r2['elapsed']:.4f}s")

# Same prompt, different temperature → MISS
r3 = chat_with_cache(prompt, temperature=0.9)
print(f"temp=0.9  →  Cached: {r3['cached']}  |  Time: {r3['elapsed']:.4f}s")

print(f"\n📊 Cache stats — Size: {cache.size} | Hits: {cache.hits} | Misses: {cache.misses}")

temp=0.0  →  Cached: False  |  Time: 1.5020s
temp=0.0  →  Cached: True  |  Time: 0.0001s
temp=0.9  →  Cached: False  |  Time: 1.3168s

📊 Cache stats — Size: 3 | Hits: 2 | Misses: 3


This is intentional. With `temperature=0.0`, the output is deterministic — caching makes perfect sense. With `temperature=0.9`, the user likely *wants* varied output, so we shouldn't return a stale cached answer.

> **Design tip:** For creative/high-temperature use cases, you might skip caching entirely or set a short TTL (time-to-live).

## Step 5 — Multiple Requests with Cache Stats

Let's simulate a realistic workload: several requests where some repeat.

In [24]:
# Reset cache for a clean demo
cache = LLMCache()

requests_to_make = [
    "What is REST? One sentence.",       # new
    "What is GraphQL? One sentence.",     # new
    "What is REST? One sentence.",         # repeat → HIT
    "What is gRPC? One sentence.",         # new
    "What is GraphQL? One sentence.",      # repeat → HIT
    "What is REST? One sentence.",         # repeat → HIT
]

total_time = 0
for i, req in enumerate(requests_to_make, 1):
    result = chat_with_cache(req)
    total_time += result['elapsed']
    status = "HIT  ✅" if result['cached'] else "MISS ❌"
    print(f"  Request {i}: {status}  {result['elapsed']:.4f}s  ← {req}")

print(f"\n📊 Final stats:")
print(f"   Cache size: {cache.size} entries")
print(f"   Hits: {cache.hits} | Misses: {cache.misses}")
print(f"   Hit rate: {cache.hit_rate:.0%}")
print(f"   Total time: {total_time:.2f}s")

  Request 1: MISS ❌  1.3596s  ← What is REST? One sentence.
  Request 2: MISS ❌  1.0950s  ← What is GraphQL? One sentence.
  Request 3: HIT  ✅  0.0000s  ← What is REST? One sentence.
  Request 4: MISS ❌  1.2591s  ← What is gRPC? One sentence.
  Request 5: HIT  ✅  0.0001s  ← What is GraphQL? One sentence.
  Request 6: HIT  ✅  0.0001s  ← What is REST? One sentence.

📊 Final stats:
   Cache size: 3 entries
   Hits: 3 | Misses: 3
   Hit rate: 50%
   Total time: 3.71s


Half the requests were served from cache — that's 50% fewer API calls, meaning **50% less cost and dramatically lower average latency**.

## Cache Design Decisions

When adding caching to a real LLM application, consider:

| Decision | Options | Tradeoff |
|----------|---------|----------|
| **Cache key** | Exact match (prompt + params) vs. semantic similarity | Exact is simple; semantic catches paraphrases but is complex |
| **Storage** | In-memory dict vs. Redis vs. database | In-memory is fast but lost on restart; Redis persists across deploys |
| **Eviction** | No limit vs. LRU vs. TTL | Unbounded caches grow forever; LRU/TTL prevent memory issues |
| **When to cache** | Always vs. only low-temperature | High-temperature responses are meant to vary — caching defeats the purpose |

The in-memory dictionary we built here is the simplest approach and works well for single-process applications. For production multi-process deployments, you'd swap in Redis or a similar shared store — the cache logic stays the same.

## 🎯 Summary

### Key Takeaways

1. **Caching eliminates redundant API calls** — identical requests are served instantly from stored results, saving both cost and latency
2. **Cache keys must capture all parameters that affect the response** — message, temperature, and model together determine whether two requests are "the same"
3. **Cache hit rates depend on your workload** — applications with repetitive queries (FAQ bots, classification) benefit most from caching
4. **Temperature matters for caching decisions** — deterministic (low-temperature) calls are ideal for caching; high-temperature calls may be better left uncached